# Paso 1 · Inspeccionar el clasificador antes de exportarlo

**Meta:** entrenar la regresión logística (LR) sobre TODA la data y *mirar* qué guarda adentro. Todavía no exportamos — primero entendemos las piezas.

## Por qué
La LR vive en Python (scikit-learn). El worker en vivo es JavaScript → no puede correr sklearn.
La solución no es montar un servidor Python (over-engineering), sino **llevar los pesos a JS**.
Una LR entrenada es solo números: `logits = W · x + b → argmax`. Eso es un dot product,
justo lo que el worker ya hace para el cosine. Y `x` = el embedding que ya calcula para retrieval.

## Las 3 piezas que definen el "cerebro" de la LR
| Objeto | Qué es | Forma esperada |
|---|---|---|
| `coef_` | matriz de pesos **W** — una fila por clase | `(n_clases, 768)` |
| `intercept_` | sesgo **b** — uno por clase | `(n_clases,)` |
| `classes_` | nombres de las clases, **en orden** | lista |

⚠️ El orden de `classes_` es sagrado: en JS, `argmax(logits)` da un índice → `classes_[índice]` es la etiqueta. Si se desalinea, predice mal.

## Gotcha crítico (para que el test pase después)
La LR se entrena sobre embeddings **crudos** (ver `features.py`, no normaliza).
El worker, para el cosine, **sí** normaliza. → En JS hay que alimentar la LR con el embedding **crudo**, no el normalizado. Mantené los dos.

## Qué sigue
Paso 2: volcar `coef_`, `intercept_`, `classes_` de cada etiqueta a un JSON (estilo `kb_vectors.json`) para bundlearlo en el worker.

In [1]:
# OBJETIVO: entrenar la LR sobre TODA la data y VER qué guarda adentro.
# Todavía no exportamos nada — solo miramos las piezas.

from src.features import build_features            # trae X (embeddings) + tickets
from sklearn.linear_model import LogisticRegression

# Las 5 etiquetas que predecimos por ticket
LABELS = ["topic", "type", "priority", "routing", "sentiment"]

# X = matriz (n, 768): un embedding por ticket.  tickets = lista con sus etiquetas.
X, tickets = build_features()
print("Tickets:", len(tickets), "| dims por embedding:", X.shape[1])

for label in LABELS:
    y = [t[label] for t in tickets]               # la respuesta correcta de cada ticket
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, y)                                 # aprende de TODOS los tickets
    # coef_ = pesos (W) · intercept_ = sesgo (b) · classes_ = nombres de clases
    print(label, "->", clf.coef_.shape, clf.intercept_.shape, list(clf.classes_))

FileNotFoundError: [Errno 2] No such file or directory: 'data/tickets_v2.jsonl'